In [1]:
import pandas as pd
df=pd.read_csv("data/titanic_clean.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,HasCabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0


In [2]:
print("流失值：",df.isnull().sum().sum())
print("形状：",df.shape)
print("文字列：",[c for c in df.columns if df[c].dtype=="object"])

流失值： 0
形状： (889, 12)
文字列： ['Name', 'Sex', 'Ticket', 'Embarked']


In [3]:
#编码sex   
df["Sex_num"]=df["Sex"].map({"male":0,"female":1})#将male改成0，female改成1
df[["Sex","Sex_num"]].head()

,Sex,Sex_num
0,male,0
1,female,1
2,female,1
3,female,1
4,male,0


In [4]:
#编码Embarked（独热编码）
dummies=pd.get_dummies(df["Embarked"],prefix="Emb")  #get_dummies作用将Embarked拆成3个0/1列  prefix="Emb"将新列改名为Emb_C/Q/S
df=pd.concat([df,dummies],axis=1)#把新列横着接回原表
print(dummies.head())

   Emb_C  Emb_Q  Emb_S
0  False  False   True
1   True  False  False
2  False  False   True
3  False  False   True
4  False  False   True


In [5]:
df["Title"]=df["Name"].str.extract(r",\s*([^.]+)\.")#提取称号，从Name中提取Mr/Mrs/Miss
#r",\s*([^.]+)\." 中1.r为原始字符串，保证\s\.不被当成转义字符  2.，先找到一个逗号  3.\s*都好后跟任意个空格\s空格*=0个或多个  4.([^.]+)代表要提取的部分括号代表捕获，[*.]是非句点字符，+代表一个或是多个  5.\.最后一个句点结束
df["Title"].value_counts()  #计数，记出现的次数

Title
Mr              517
Miss            181
Mrs             124
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64

In [6]:
df.groupby("Title")["Survived"].agg(["mean","count"]).round(3).sort_values("mean",ascending=False)
#df.groupby("Title") 按称号分组
#["Survived"]每组只看Survived这一列
#.agg(["mean","count"]) 每组算两个数：平均值+个数
#.round(3)保留三位小数
#.sort_values("mean",ascending=False)按生还率从高到低排序

,mean,count
Title,,
the Countess,1.000,1
Mlle,1.000,2
Sir,1.000,1
Ms,1.000,1
Lady,1.000,1
Mme,1.000,1
Mrs,0.790,124
Miss,0.696,181
Master,0.575,40


In [7]:
rare=["Dr","Rev","Major","Col","Mlle","the Countess","Capt","Ms","Sir","Lady","Mme","Don","Jonkheer"]
df["Title"]=df["Title"].replace(rare,"Rare")

In [8]:
df.groupby("Title")["Survived"].agg(["mean", "count"]).round(3).sort_values("mean", ascending=False)

,mean,count
Title,,
Mrs,0.790,124
Miss,0.696,181
Master,0.575,40
Rare,0.444,27
Mr,0.157,517


In [9]:
#造 FamilySize
df["FamilySize"]=df["SibSp"]+df["Parch"]+1  #SibSp同行的家人+同行的父母子女+自己一个人
df.groupby("FamilySize")["Survived"].agg(["mean", "count"]).round(3)

,mean,count
FamilySize,,
1,0.301,535
2,0.553,161
3,0.578,102
4,0.724,29
5,0.200,15
6,0.136,22
7,0.333,12
8,0.000,6
11,0.000,7


In [10]:
#组合特征的威力
df.pivot_table(values="Survived",index="Sex",columns="Pclass",aggfunc="mean").round(3)

Pclass,1,2,3
Sex,,,
female,0.967,0.921,0.500
male,0.369,0.157,0.135


In [11]:
features=["Pclass","Sex_num","Age","SibSp","Parch","Fare","HasCabin","Emb_C","Emb_Q","Emb_S"]
X=df[features]
y=df["Survived"]

print("特征集：",X.shape)
print("全是数字？",not any(X.dtypes=="object"))

特征集： (889, 10)
全是数字？ True


1	为什么 Embarked 要用独热编码，而 Pclass 可以直接用 1/2/3？
2	Master（小男孩）生还率 57.5%，比 Mr 高 4 倍 —— 为什么？
3	家庭规模为什么是"先升后降"而不是"越多越好"？


1.因为embarked它有三个值用不了map转化为0和1，Pclass不知道是为什么
2.因为在遇难时总会让妇女儿童先走，男人殿后
3.不知道

修正
1.「有序」的类别 → 用数字编码（1/2/3）
「无序」的类别 → 用独热编码（拆成多列 0/1）
2.Title 从 Name 里，提取出了 Sex 无法提供的额外信息 —— 年龄 / 社会角色。
"妇女和儿童优先"这条规则里的"儿童"，只有 Title 能识别出来。
3.第一层：表面原因 —— 大家庭大多在三等舱"大家庭生还率低"，一部分原因是"他们穷，住在最底层"
第二层：但控制舱位后，差距依然存在 ⭐同样是三等舱，大家庭的生还率还是只有小家庭的 1/5。
说明"家庭规模"确实有独立的影响，不只是"因为穷"
家人互助	生还率 ↑	有人帮忙抢位置、互相照应
决策拖累	生还率 ↓	一家人不愿分开，要"找到所有人再走"，耽误了登船时间


2~4 人：两者平衡得最好 → 72.4%（小家庭最高）
5 人以上：拖累压倒帮助 → 13.6%
1 人：没有互助 → 30.1%